# LAB | Abstractive Question Answering

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [ ]:
!pip install -qU datasets pinecone-client==3.1.0 sentence-transformers torch transformers tqdm

# Load and Prepare Dataset

Our source data will be taken from the `wikimedia/wikipedia` dataset, which contains Wikipedia articles.
We will extract 50,000 passages from sections that contain the word "History".

> **Note:** The original `vblagoje/wikipedia_snippets_streamed` dataset used a loading script that is no longer supported by the `datasets` library (it raises `RuntimeError: Dataset scripts are no longer supported`). We use `wikimedia/wikipedia` (20231101.en) instead — a standard Parquet-based dataset with the same Wikipedia content.

In [ ]:
from datasets import load_dataset

# load the Wikipedia dataset in streaming mode so we don't download all 20+ GB
wiki_data = load_dataset(
    'wikimedia/wikipedia',
    '20231101.en',
    split='train',
    streaming=True
).shuffle(seed=960)

In [ ]:
# show the contents of a single document in the dataset
sample = next(iter(wiki_data))
print('Keys:', list(sample.keys()))
print('Title:', sample['title'])
print('Text preview:', sample['text'][:300])

The `wikimedia/wikipedia` dataset stores full articles as a single `text` field.
We split each article into paragraphs and keep only paragraphs from sections whose heading contains "History".
This gives us the same kind of passage-level documents as the original dataset.

In [ ]:
def extract_history_passages(article):
    """Split a Wikipedia article into paragraphs and return those
    that belong to a section whose heading contains 'History'."""
    passages = []
    lines = article['text'].split('\n')
    current_section = ''
    buffer = []

    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Section headings in the wikimedia dump are plain-text lines
        # surrounded by == markers
        if line.startswith('==') and line.endswith('=='):
            # flush previous section buffer
            if 'history' in current_section.lower() and buffer:
                for para in buffer:
                    if len(para) > 80:  # skip very short lines
                        passages.append({
                            'article_title': article['title'],
                            'section_title': current_section.strip('= '),
                            'passage_text': para
                        })
            current_section = line
            buffer = []
        else:
            buffer.append(line)

    # flush the last section
    if 'history' in current_section.lower() and buffer:
        for para in buffer:
            if len(para) > 80:
                passages.append({
                    'article_title': article['title'],
                    'section_title': current_section.strip('= '),
                    'passage_text': para
                })
    return passages

In [ ]:
from tqdm.auto import tqdm

total_doc_count = 50000
docs = []

# iterate through the streaming dataset and collect history passages
with tqdm(total=total_doc_count, desc='Collecting history passages') as progress_bar:
    for article in wiki_data:
        passages = extract_history_passages(article)
        for passage in passages:
            docs.append(passage)
            progress_bar.update(1)
            if len(docs) >= total_doc_count:
                break
        if len(docs) >= total_doc_count:
            break

print(f'Collected {len(docs)} history passages')

In [ ]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). Get a free API key from [app.pinecone.io](https://app.pinecone.io/).

In [ ]:
import os
from pinecone import Pinecone

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.environ.get('PINECONE_API_KEY') or 'YOUR_PINECONE_API_KEY_HERE'

# configure client
pc = Pinecone(api_key=api_key)

In [ ]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

In [ ]:
# give our index a meaningful name
index_name = 'abstractive-question-answering'

In [ ]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # create index with cosine similarity and 768 dimensions (MPNet output size)
    pc.create_index(
        index_name,
        dimension=768,
        metric='cosine',
        spec=spec
    )
    # wait for index to be ready
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to the index
index = pc.Index(index_name)

# ensure the stats show a fresh index
index.describe_index_stats()

# Initialize Retriever

The retriever generates embeddings for all historical passages (context vectors) and for our questions (query vectors). We use a SentenceTransformer model based on Microsoft's MPNet which is optimized for cosine similarity between queries and documents.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from HuggingFace
retriever = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device)
retriever

# Generate Embeddings and Upsert

We generate embeddings for the context passages in batches and upload them to Pinecone. Each document is stored with an id, context embedding, and metadata (article title, section title, passage text).

In [ ]:
# we will use batches of 64
batch_size = 64

# create embeddings for the passage_text and upsert into Pinecone with metadata
for i in tqdm(range(0, len(df), batch_size), desc='Upserting to Pinecone'):
    # find end of batch
    batch_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:batch_end]
    # generate embeddings for the passage_text column
    embeddings = retriever.encode(batch['passage_text'].tolist()).tolist()
    # build list of (id, vector, metadata) tuples
    vectors = [
        (
            str(row_idx),
            embedding,
            {
                'article_title': row['article_title'],
                'section_title': row['section_title'],
                'passage_text': row['passage_text']
            }
        )
        for row_idx, (_, row), embedding in zip(range(i, batch_end), batch.iterrows(), embeddings)
    ]
    # upsert into pinecone index
    index.upsert(vectors=vectors)

# check that we have all vectors in index
index.describe_index_stats()

# Initialize Generator

We use ELI5 BART for the generator — a Sequence-To-Sequence model trained on the 'Explain Like I'm 5' dataset. The input is a concatenation of the query and context passages separated by `<P>` tokens:

> question: What is a sonic boom? context: \<P> A sonic boom is a sound associated with shock waves...

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

In [ ]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    query_embedding = retriever.encode([query]).tolist()
    # search pinecone index for context passages with the answer
    search_results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)
    return search_results

In [ ]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context_passages = [f"<P> {match['metadata']['passage_text']}" for match in context]
    # concatenate all context passages
    context_str = ' '.join(context_passages)
    # concatenate the query and context passages
    formatted_query = f"question: {query} context: {context_str}"
    return formatted_query

In [ ]:
# test the helper functions
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

In [ ]:
from pprint import pprint

# format the query in the form the generator expects
formatted = format_query(query, result["matches"])
pprint(formatted)

In [ ]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [ ]:
generate_answer(formatted)

Let's run some more queries.

In [ ]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
formatted_query = format_query(query, context["matches"])
generate_answer(formatted_query)

In [ ]:
# confirm the answer by checking the contexts used
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers:

In [ ]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
formatted_query = format_query(query, context["matches"])
generate_answer(formatted_query)

In [ ]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

Let's finish with a final few questions:

In [ ]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
formatted_query = format_query(query, context["matches"])
generate_answer(formatted_query)

In [ ]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
formatted_query = format_query(query, context["matches"])
generate_answer(formatted_query)

In [ ]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
formatted_query = format_query(query, context["matches"])
generate_answer(formatted_query)

As we can see, the model can generate some decent answers.